## Localizing FathomNet to a New Dataset Using YOLOv11
### Learning Objectives



By the end of this lesson, you will be able to:



1. Understand how to adapt the FathomNet dataset to a new dataset.

2. Utilize one of MBARI's foundational fathomnet models (MBARI 315k) for inference.

3. Train a YOLOv11 model using the localized dataset with adjusted parameters for optimal performance.

4. Experiment with prediction and tracking modes.


---

### Introduction to FathomNet and MBARI's Foundational Models



FathomNet is a large dataset designed for marine imagery analysis, and MBARI has developed many models from their dataset. The two most helpful for "localizing" a new dataset are:



1. **Megalodon**: A region of interest (ROI) detector with a single class, "object."

   [Megalodon Model](https://huggingface.co/FathomNet/megalodon)



2. **MBARI 315k**: A taxonomy-based object detector trained on a large-scale dataset.

   [MBARI 315k Model](https://huggingface.co/FathomNet/MBARI-315k-yolov8)



**Localization** in this context refers to the process of adapting a general model, like those trained on the FathomNet dataset, to work effectively on a specific dataset. This is done by utilizing the pre-trained weights from these models and fine-tuning them with new labels and annotations from your dataset. By starting with pre-trained models, significant time is saved because:



- The models already encode a large amount of knowledge about marine imagery, reducing the need for extensive initial training.

- Localization allows the transfer of this learned information to a new dataset, which may have unique characteristics, by focusing on fine-tuning rather than training from scratch.



These models provide a foundation for efficient training and allow researchers to quickly generate results tailored to their specific needs. Up until recently, the only starting checkpoints for localization were based on large-scale datasets like COCO, which often contain no relevant data for specific scientific domains like marine imagery. Having models like Megalodon and MBARI 315k, whose weights are already tuned to the same domain, enables significantly better performance and reduces the time required to adapt a model. This domain-specific starting point allows researchers to achieve meaningful results without needing to train entirely from scratch. For our purposes we will be using the 315K model in this activity as it has more relavence to our dataset. If your dataset is particularly distinct from the deep sea benthos, it may make more sense to start with the Megalodon model as that is what Fathomnet is primarily trained on.



---
### Dataset Preparation and Inference



For this lesson, we will use a new dataset: a 38-minute ROV transect near a methane seep that has been compressed from its original resolution for ease of import and predictions. ROV transects are often used to survey an area and its ecosystem, and these videos are traditionally analyzed manually or qualitatively. However, these transects can often span hours of footage and require significant labor to analyze effectively.



To make localization practical for real-world use, instead of processing an entire dataset at once, we recommend generating a subset of videos with representative classes. This can be achieved by skimming through the dataset and identifying unique or diverse instances in the video or images. A general rule of thumb is to use approximately 20 minutes of video footage or 2000 still images that encompass a variety of classes representative of the entire dataset. For our purposes, we will be subsetting this video into every 32nd frame, providing roughly 2000 images to work with. This subset size is manageable for one person to analyze and provides a quick enough turnaround to make localization worthwhile.



By working with such subsets, the process becomes more efficient while still allowing for effective adaptation of the FathomNet models to the dataset.



#### Run Initial Predictions



Use the `ultralytics` library to run predictions with both models. Passing the `save_txt=True` parameter is essential as it saves the text annotations produced in a format that is easy to import and modify:



In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics

In [ ]:
import os
import cv2
from ultralytics import YOLO

# Create a folder for the sampled frames
subset_folder = "frames"
os.makedirs(subset_folder, exist_ok=True)

# Input video file
video_path = "/content/transect_compressed.mp4"

# Sample every 32nd frame from the video
cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_rate = int(cap.get(cv2.CAP_PROP_FPS))
sample_rate = 32
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % sample_rate == 0:
        frame_filename = os.path.join(subset_folder, f"frame_{frame_count}.jpg")
        cv2.imwrite(frame_filename, frame)

    frame_count += 1

cap.release()

print(f"Sampled frames saved in folder: {subset_folder}")

'''
# Load the Megalodon model
megalodon_model = YOLO("https://huggingface.co/FathomNet/megalodon/resolve/main/best.pt")

# Run inference on the sampled frames with Megalodon
megalodon_model.predict(
    source=subset_folder,
    save_txt=True,
    imgsz=1024,
    conf=0.10,
    iou=0.5,
    agnostic_nms=True
)
'''

# Load the MBARI 315k model
mbari_model = YOLO("https://huggingface.co/FathomNet/MBARI-315k-yolov8/resolve/main/mbari_315k_yolov8.pt")

# Run inference on the sampled frames with MBARI 315k
mbari_model.predict(
    source=subset_folder,
    save_txt=True,
    imgsz=1024,
    conf=0.10,
    iou=0.5,
    agnostic_nms=True
)


---

#### Localizing Annotations

Next, we want to work on localizing the predicted annotations to our class nomenclature. For this example, we will be taking the annotations given by the 315k model in taxonomic format and collapsing them into broader categories, referred to as ecological tiers. This step is done to speed up the activity and act as a proof of concept. However, always remember to consider your research question and the degree of specificity you need for your classes.

Below is a guide for the four representative tiers present in this transect. This is not an exhaustive list of taxa that fit these descriptions but rather a representative sample. The four tiers are:

Sessile Epifauna: Organisms that are attached to the substrate, such as anemones and sponges.

Motile Epifauna: Organisms capable of moving that primarily live on the substrate, such as sea urchins, cucumbers and  stars.

Demersal: Organisms that live near or on the seafloor but are capable of swimming, like benthic fish.

Planktonic: Organisms that drift in the water column, such as euphasiids and other plankton.

:::{figure} images/Epifauna.png
:name: Broad Classes
OOI/UW/NSF Carter 2025
:::
To proceed:

1. Go into an annotation manager like Roboflow and upload the FathomNet labelmap along with your dataset. (The labelmap can be accessed on the [315 huggingface](https://huggingface.co/FathomNet/MBARI-315k-yolov8) or can be automatically zipped with your dataset if you run the cell below.)

2. Luckily, the FathomNet model did most of the heavy lifting and should have predicted relatively close to the actual species. For each class, go through and review images with that class. As you go, mark down which of the four tiers each class best fits into. After reviewing all classes, bulk reassign the classes into the four tiers. Then, go through and check if anything was missed.

3. Once complete, export your dataset as a ZIP file in YOLOv11 format for further processing in the colab environment. Ensure that your splits are balanced per class before exporting! A test set will not be necessary.

In [ ]:
import zipfile
import os
import urllib.request

def zip_folders_and_file(folder_paths, additional_file_url, output_filename):

    additional_file_path = "config.yaml"
    urllib.request.urlretrieve(additional_file_url, additional_file_path)

    with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for folder in folder_paths:
            for root, _, files in os.walk(folder):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, start=folder)
                    zipf.write(file_path, arcname)
        zipf.write(additional_file_path, os.path.basename(additional_file_path))

folders_to_zip = ["/content/frames", "/content/runs/detect/predict/labels"]
additional_file_url = "https://huggingface.co/FathomNet/MBARI-315k-yolov8/resolve/main/config.yaml?download=true"
output_zip = "/content/315k.zip"
zip_folders_and_file(folders_to_zip, additional_file_url, output_zip)


In [ ]:
from google.colab import files

files.download('/content/315k.zip')  # Download 315k.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Localizing the Model

Now that you have a finalized four-class dataset, it’s time to train a model. Use MBARI 315k as a checkpoint to import useful weights and override the old class labels with your new class labels to create a robust model tailored to your dataset.

For this training, we will select the following parameters:

Data Configuration: data.yaml, which includes the paths to your train and validation datasets.

Epochs: Set to 100 to ensure sufficient learning while avoiding overfitting.

Image Size: 1024 to balance detail and computational efficiency.

Patience: 10, to allow the model to terminate training early if no improvement is seen.

Plots: Enabled (plots=True) to generate visualizations of the training progress.

Below is the code to initiate training:

In [ ]:
from ultralytics import YOLO

# Load a pretrained model
model = YOLO("mbari_315k_yolov8.pt")

# Train the model
results = model.train(data="data.yaml", epochs=100, imgsz=1024, plots=True, patience=10)

### Track in Zone

Tracking objects within a specific zone is crucial for analyzing ROV and other transects where the objects you are interested in pass by the camera. By focusing on a defined region, researchers can ensure accurate detection and counting of marine organisms or features while minimizing noise from irrelevant areas. This method helps standardize data collection, enabling more reliable comparisons across different transects and improving ecological assessments of underwater environments.


In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# User Config
# Path to your weights 
model_path = "/content/best.pt" 
image_path = "/content/testframe.png"

# List of (Confidence, IoU) 
# Format: (conf, iou)
threshold_settings = [
    (0.01, 0.5),  # Low conf, standard IoU
    (0.5, 0.5),  # High conf, standard IoU
    (0.1, 0.1),  # Low conf, Low IoU 
    (0.1, 0.9)   # Low conf, High IoU 
]

# Execution
model = YOLO(model_path)
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Setup subplots
fig, axes = plt.subplots(1, len(threshold_settings), figsize=(20, 6))
if len(threshold_settings) == 1: axes = [axes]

print(f"Running inference on {image_path} with {len(threshold_settings)} different settings...")

for ax, (conf, iou) in zip(axes, threshold_settings):
    # Run inference with specific parameters
    results = model.predict(image_path, conf=conf, iou=iou, verbose=False)
    
    # Plot results
    res_plotted = results[0].plot()
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
    
    ax.imshow(res_rgb)
    ax.set_title(f"Conf: {conf} | IoU: {iou}", fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px
import pandas as pd
from ultralytics import YOLO

# User Config
model_path = "/content/best.pt" 
image_path = "/content/testframe.png"

# Confidence and IoU pairs
test_settings = [
    (0.1, 0.5), (0.2, 0.5), (0.3, 0.5), 
    (0.1, 0.7), (0.3, 0.7)
]

# Execution
model = YOLO(model_path)
data = []

print("Gathering data points...")
for conf_val, iou_val in test_settings:
    results = model.predict(image_path, conf=conf_val, iou=iou_val, verbose=False)
    
    for box in results[0].boxes:
        # Calculate Box Area (Width * Height)
        # box.xywh returns (x_center, y_center, width, height)
        w = box.xywh[0][2].item()
        h = box.xywh[0][3].item()
        area = w * h
        
        conf_score = box.conf[0].item()
        cls_id = int(box.cls[0].item())
        class_name = model.names[cls_id]
        
        data.append({
            "Setting": f"Conf: {conf_val} / IoU: {iou_val}",
            "Area (px²)": area,
            "Confidence": conf_score,
            "Class": class_name
        })

if data:
    df = pd.DataFrame(data)
    fig = px.scatter(
        df, 
        x="Area (px²)", 
        y="Confidence", 
        color="Setting", 
        symbol="Class",
        title="Prediction Confidence vs. Bounding Box Area",
        hover_data=["Class", "Area (px²)"],
        template="plotly_dark",
        width=1000, height=600
    )
    fig.show()
else:
    print("No detections were made with the provided settings.")


In [ ]:
cap = cv2.VideoCapture("/content/transectx163s.mp4")
assert cap.isOpened(), "Error reading video file"
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH,
                                       cv2.CAP_PROP_FRAME_HEIGHT,
                                       cv2.CAP_PROP_FPS))

In [ ]:
video_writer = cv2.VideoWriter("counting1.avi",
                               cv2.VideoWriter_fourcc(*"mp4v"),
                               fps, (w, h))

In [ ]:
top_region_y = int(h * (2/3))

region_points = [
    (0, top_region_y),      # Top-Left of zone
    (w, top_region_y),      # Top-Right of zone
    (w, h),                 # Bottom-Right of zone
    (0, h)                  # Bottom-Left of zone
]

In [ ]:
counter = solutions.ObjectCounter(
    region=region_points,  
    model="/content/best.pt",
    conf=0.10,
    iou=0.5,
)

In [ ]:
import numpy as np

while cap.isOpened():
    success, im0_original = cap.read() # Read original frame into a new variable
    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    processed_results = counter(im0_original) 

    if hasattr(processed_results, 'plot') and callable(processed_results.plot):
        processed_frame = processed_results.plot()
    else:

        processed_frame = im0_original
        print(f"Warning: Could not plot from ObjectCounter results. Writing original frame.")

    if isinstance(processed_frame, np.ndarray):
        video_writer.write(processed_frame)
    else:
        print(f"Warning: ObjectCounter returned an invalid frame type ({type(processed_results)}) for a frame even after attempting to plot. Writing original frame.")
        video_writer.write(im0_original)

cap.release()
video_writer.release()
